# C-16 Stratified Model Evaluation

**Control:** C-16 Stratified Model Evaluation, MAK-AI Responsible AI Toolkit.

**Learning goal:** decide between one pooled model and separate (stratified) models by evaluating each on every group; handle groups with little data; and avoid leakage when you cross-validate.

**Time to run:** about 5 minutes on an ordinary laptop with two cores. No GPU.

**Licence:** this notebook is Apache-2.0.

**Example data:** MasakhaNEWS (Adelani and others, 2023), Hugging Face dataset `masakhane/masakhanews`, revision fa3b5ff (the dataset's state on 3 Dec 2025), licence tag afl-3.0, news headlines and articles in four African languages: Luganda (`lug`), Swahili (`swa`), Hausa (`hau`) and Yoruba (`yor`), with seven topic labels (business, entertainment, health, politics, religion, sports, technology), not all of which occur in every language: Luganda and Yoruba have five each. From https://huggingface.co/datasets/masakhane/masakhanews. It is credited here because the licence asks for attribution.

**A caveat before you start:** only the release licence of the dataset was checked. The rights in the news texts underneath were not, so keep the data on your own machine and do not republish it. The notebook uses four of the sixteen languages to keep the run short, and a topic classifier is a stand-in for whatever your model does.

## Linked risk areas and questions

Generated from content 5.0.0. Do not edit by hand.

Risk areas: 1

- IP-02: Aggregation Bias

Questions: 7: Q-PP-23, Q-IP-02, Q-IP-04, Q-IP-06, Q-IP-13, Q-IP-14, Q-IP-19

## When to use this notebook

Use it when you are answering questions about whether one model serves every group equally well and your status is **Not yet** or **Partly**, especially with the reason code `ny_needs_skills` ("Needs skills or guidance the team does not have yet"). It helps with:

- `Q-IP-02`: are the training, validation and test datasets representative of the same underlying population?
- `Q-IP-04`: have you calculated performance metrics separately for each relevant subgroup?
- `Q-IP-13`: which additional evaluation methods have you applied (cross-validation, error analysis by subgroup)?
- `Q-IP-14` and `Q-IP-19`: does aggregating data hide differences between subgroups, and do the model's errors fall differently on different groups?

A team may also attach the summary to `Q-PO-12` (performance across languages after deployment), although the framework does not link that question to this control.

You need text (or other records) in several groups such as languages, with a label for each, and a train and test split for each group.

In [ ]:
import hashlib
import importlib.metadata
import io
import os
import platform
import tempfile
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from fairlearn.metrics import MetricFrame
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.utils import resample

SEED = 20261030  # the one random seed; every random step below uses it
pd.set_option('display.width', 150)
pd.set_option('display.max_columns', 30)

PACKAGES = ['pandas', 'numpy', 'scikit-learn', 'fairlearn', 'matplotlib']

print('Python', platform.python_version())
for _package in PACKAGES:
    print(_package, importlib.metadata.version(_package))

In [ ]:
# Pinned to one revision of the dataset, so the files cannot change under you. One address and one SHA-256 per file.
DATA_URL_BASE = 'https://huggingface.co/datasets/masakhane/masakhanews/resolve/fa3b5fff8a91d187bf0c5900a39c4271d08cf7fe/data/'
DATA_SHA256 = {
    ('lug', 'train'): '48a4647c0fb3c61a2ddbac3c36e6d2e4baf41bad32b49b4b5fa555c172e6f3dd',
    ('lug', 'dev'): '2c7eae3da302aec81649590c178e55961cb2ef06638352c7218e52735bbaf892',
    ('lug', 'test'): '4f9ea6f76f596e875b4f6cc17aa5758fa6f32cebe113a33a9da4d65d1cd6a43f',
    ('swa', 'train'): '90f6400a91164f734a37157acb2b3a3f06addd0ba818cb508cb74f4f926e5032',
    ('swa', 'dev'): '3bdbcea3c4a2dc878181c24ce2570ba49fe6a2d90bc8d604d22f35d317d02054',
    ('swa', 'test'): '6d7e076f2fb53e46c7b2ed855fd77aa5e3351435e2bddafcd07cceeceada61aa',
    ('hau', 'train'): '86aa74d6cc393e65b28d03c7770a607424e7faef4c9fe1574e7591b70698a868',
    ('hau', 'dev'): 'd117ca9ab9c61b2ea0a07b1aab7887597d4c10398eb2b3e02a4f1c87fa62f804',
    ('hau', 'test'): '7e88ffaf6a4744611c41e95df6a28c6eb3c7bb6d4f60a381e2f9d1021ef97c51',
    ('yor', 'train'): '1f00b8d500cc771c1756221d76544c0589ca806bb7c4570c64fcf3be524dae34',
    ('yor', 'dev'): '60bae7b98606a123becd9b5a8e958e350a3439f26e65f540ea5ed35343b03c89',
    ('yor', 'test'): '4645271b425351ea3ae88d763dff46303aa6aaaf7b28caa6c4508f07141c35e9',
}

CACHE_DIR = Path.home() / '.cache' / 'makrai-notebooks'


MAX_DOWNLOAD_BYTES = 64 * 1024 * 1024  # a dataset file is never larger than this; a longer body is refused


def fetch_checked(url, sha256, file_name):
    """Return the bytes of a dataset whose SHA-256 matches `sha256`.

    The file is downloaded once into the cache folder, and written there only after its hash
    matches. Every run, including the first, hashes the bytes before they are used, and a
    mismatch stops the notebook. The body is read in 1 MiB pieces and refused past MAX_DOWNLOAD_BYTES.
    `file_name` is fixed in this notebook, never taken from the address.
    """
    if not url.startswith('https://'):
        raise ValueError(f'the dataset address must be https: {url}')
    if Path(file_name).name != file_name:
        raise ValueError(f'the cache file name must be a plain name: {file_name}')

    def read_checked(stream):
        digest, pieces, size = hashlib.sha256(), [], 0
        for piece in iter(lambda: stream.read(1 << 20), b''):
            size += len(piece)
            if size > MAX_DOWNLOAD_BYTES:
                raise RuntimeError(f'{file_name}: the file is larger than {MAX_DOWNLOAD_BYTES:,} bytes; refused')
            digest.update(piece)
            pieces.append(piece)
        return b''.join(pieces), digest.hexdigest()

    path = CACHE_DIR / file_name
    if path.exists():
        with path.open('rb') as stream:
            data, found = read_checked(stream)
        source, remedy = f'the cached file {path}', f'Delete {path} and run again.'
    else:
        request = urllib.request.Request(url, headers={'User-Agent': 'makrai-notebooks'})
        try:
            with urllib.request.urlopen(request, timeout=120) as response:
                if not response.geturl().startswith('https://'):
                    raise RuntimeError(f'{url} redirected to a non-https address: {response.geturl()}')
                data, found = read_checked(response)
        except OSError as error:
            raise RuntimeError(f'could not download {url}: {error}. If you cannot reach it, save the file by hand '
                               f'as {path}; it is checked against the pinned SHA-256 either way.') from error
        source, remedy = url, 'Do not use this download; check the dataset page for a new version.'
    if found != sha256:
        raise RuntimeError(f'{file_name}: the SHA-256 of {source} is {found}, expected {sha256}. {remedy}')
    if not path.exists():
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        partial = tempfile.NamedTemporaryFile(dir=CACHE_DIR, suffix='.part', delete=False)
        try:
            with partial:
                partial.write(data)
            os.replace(partial.name, path)  # the cache never holds a half-written file, whoever else is writing
        except BaseException:
            Path(partial.name).unlink(missing_ok=True)
            raise
    return data


# To use your own files, replace the loop below with reads of your own files, one table per group and split.
frames, DATA_SHA256_USED = [], {}
for (language, split), expected_hash in DATA_SHA256.items():
    file_name = f'masakhanews-{language}-{split}.tsv'  # the cache file name is built here, never from the address
    raw = fetch_checked(f'{DATA_URL_BASE}{language}/{split}.tsv', expected_hash, file_name)
    DATA_SHA256_USED[file_name] = hashlib.sha256(raw).hexdigest()
    part = pd.read_csv(io.BytesIO(raw), sep='\t')
    part['language'], part['split'] = language, split
    frames.append(part)
df = pd.concat(frames, ignore_index=True)
print(f'{len(df):,} articles in {df.language.nunique()} languages; {len(DATA_SHA256_USED)} files, each checked against its SHA-256')
print(df.groupby(['language', 'split']).size().unstack().to_string())

In [ ]:
# ADAPT HERE. Edit only this cell to run the notebook on your own data. Every later cell reads only these names.
GROUP_COLUMN = 'language'            # the column that says which group each row belongs to
SPLIT_COLUMN = 'split'               # the column that says whether a row is for development or for the final test
TEST_SPLIT = 'test'                  # the value of SPLIT_COLUMN that marks the held-out test rows
TEXT_COLUMNS = ['headline', 'text']  # the columns joined into the text the model reads
LABEL = 'category'                   # the column that holds the true label
DUPLICATE_KEY = 'headline'           # rows with the same value here are the same story; they must not straddle a split
MIN_GROUP_SIZE = 100                 # a group with fewer test rows than this is flagged, not interpreted
N_BOOTSTRAP = 200                    # resamples for the intervals
N_FOLDS = 5                          # folds for the cross-validation in Step 5
TRAIN_FRACTIONS = [0.1, 0.25, 0.5, 1.0]   # shares of the smallest group's data used in Step 6
MAX_FEATURES = 20000                 # how many words the text model keeps

### Step 1: set up one text model and the groups

**What this does.** It builds one recipe for the model: **TF-IDF** (term frequency, inverse document frequency) turns each text into numbers by counting words and giving more weight to words that are distinctive, then a logistic regression predicts the label from them. The same recipe is used for every model below, so differences come from how the data is split between models, not from a different algorithm. The text is the headline and the article joined together. The data is split by the file's own marks into a **development** part (for fitting and for choosing) and a **test** part (looked at once, at the end).

**What it shows.** The groups, their sizes and the labels.

**What it does not show.** That this recipe is the best. It is a plain baseline chosen to be quick, and a stronger model would shift every number, though not the method.

In [ ]:
df['_text'] = df[TEXT_COLUMNS].fillna('').astype(str).agg(' '.join, axis=1)
is_test = df[SPLIT_COLUMN] == TEST_SPLIT
development, held_out = df[~is_test].reset_index(drop=True), df[is_test].reset_index(drop=True)
GROUPS = sorted(df[GROUP_COLUMN].unique())
smallest_group = development[GROUP_COLUMN].value_counts().idxmin()


def new_model():
    return make_pipeline(TfidfVectorizer(max_features=MAX_FEATURES, sublinear_tf=True, min_df=2), LogisticRegression(max_iter=1000))


def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


print(f'development rows per group:\n{development[GROUP_COLUMN].value_counts().to_string()}')
print(f'\nsmallest group: {smallest_group}; labels: {sorted(df[LABEL].unique())}')

### Step 2: the pooled model, and the separate models

**What this does.** It fits **one pooled model** on the development rows of every group together, and **one separate model for each group** on that group's development rows only. Both are scored on every group's held-out test rows with Fairlearn's `MetricFrame`, which computes the same metrics for each group:

- **accuracy**: the share of articles given the right label.
- **macro F1**: the average, over labels, of how well each label is found (F1 combines precision and recall for one label). It gives a rare label as much weight as a common one, so it does not let a model hide behind the big topics.

**What it shows.** For each group, whether it does better in the pooled model (which can borrow strength from the others) or in its own.

**What it does not show.** Whether a difference is real (Step 3 adds intervals), or why. Languages share some words and topics and not others, and a pooled model also shares errors.

In [ ]:
started = time.time()
pooled = new_model().fit(development['_text'], development[LABEL])
pooled_prediction = pd.Series(pooled.predict(held_out['_text']), index=held_out.index)

separate_prediction = pd.Series(index=held_out.index, dtype=object)
for group in GROUPS:
    rows = development[GROUP_COLUMN] == group
    model = new_model().fit(development.loc[rows, '_text'], development.loc[rows, LABEL])
    test_rows = held_out[GROUP_COLUMN] == group
    separate_prediction[test_rows] = model.predict(held_out.loc[test_rows, '_text'])

METRICS = {'accuracy': accuracy_score, 'macro F1': macro_f1}
evaluation = {}
for name, prediction in {'pooled': pooled_prediction, 'separate': separate_prediction}.items():
    frame = MetricFrame(metrics=METRICS, y_true=held_out[LABEL], y_pred=prediction, sensitive_features=held_out[GROUP_COLUMN])
    evaluation[name] = frame.by_group
    print(f'--- {name} model, test rows per group, overall accuracy {accuracy_score(held_out[LABEL], prediction):.3f} ---')
    print(frame.by_group.round(3).to_string())
print(f'(fitting and scoring took {time.time() - started:.0f} s)')

### Step 3: how sure can we be? Intervals per group

**What this does.** For each group it draws `N_BOOTSTRAP` new test sets by picking test articles at random with replacement (scikit-learn's `resample`), and on each one recomputes the macro F1 of both models and the difference between them (pooled minus separate). It keeps the range that holds the middle 95 out of every 100 draws: one range for each model, and one for the difference. Both models are measured on the same resampled articles each time, so the range of the difference is the one that says whether they differ. A group with fewer than `MIN_GROUP_SIZE` test articles is flagged.

**What it shows.** The honest size of "we do not know exactly". If the range of the difference holds zero, this data cannot tell the two designs apart for that group. If it lies wholly above or below zero, it can. Two ranges for the two models that overlap do not by themselves mean the models are alike: the same test articles are easy or hard for both, so their difference varies less than either macro F1 does.

**What it does not show.** How the models would do on new articles from a different time or site. The test articles come from the same source as the development ones.

In [ ]:
interval_rows, difference_rows = [], []
for group in GROUPS:
    rows = (held_out[GROUP_COLUMN] == group).to_numpy()
    truth = held_out.loc[rows, LABEL].to_numpy()
    guesses = {name: prediction[rows].to_numpy() for name, prediction in {'pooled': pooled_prediction, 'separate': separate_prediction}.items()}
    picks = [resample(np.arange(len(truth)), replace=True, random_state=SEED + b) for b in range(N_BOOTSTRAP)]
    draws = {name: np.array([macro_f1(truth[pick], guess[pick]) for pick in picks]) for name, guess in guesses.items()}
    for name, guess in guesses.items():
        interval_rows.append({'group': group, 'model': name, 'test articles': int(rows.sum()), 'macro F1': macro_f1(truth, guess),
                              'low': np.quantile(draws[name], 0.025), 'high': np.quantile(draws[name], 0.975), 'small group': int(rows.sum()) < MIN_GROUP_SIZE})
    difference = draws['pooled'] - draws['separate']
    difference_rows.append({'group': group, 'difference low': np.quantile(difference, 0.025), 'difference high': np.quantile(difference, 0.975)})
intervals = pd.DataFrame(interval_rows)
differences = pd.DataFrame(difference_rows)
print(intervals.round(3).to_string(index=False))
print('\nPooled minus separate, on the same resampled test articles:')
print(differences.round(3).to_string(index=False))

### Step 4: the decision table

**What this does.** It puts the two designs side by side for each group and reads the range of the difference from Step 3: **pooled looks better** if the whole range is above zero, **separate looks better** if it is wholly below zero, and **cannot tell** if it holds zero. It does not choose for you.

**What it shows.** A summary of the evidence for each group.

**What it does not show.** The decision. A pooled model is one thing to build, test, monitor and explain. Separate models are several, and a new group needs its own. A pooled model that is slightly better on average but much worse for one group may not be acceptable. A difference can be clear and still too small to matter.

In [ ]:
decision_rows = []
for group in GROUPS:
    p = intervals[(intervals.group == group) & (intervals.model == 'pooled')].iloc[0]
    s = intervals[(intervals.group == group) & (intervals.model == 'separate')].iloc[0]
    d = differences[differences.group == group].iloc[0]
    reading = 'pooled looks better' if d['difference low'] > 0 else 'separate looks better' if d['difference high'] < 0 else 'cannot tell'
    decision_rows.append({'group': group, 'training articles': int((development[GROUP_COLUMN] == group).sum()),
                          'pooled macro F1': p['macro F1'], 'separate macro F1': s['macro F1'],
                          'difference (pooled minus separate)': p['macro F1'] - s['macro F1'],
                          'difference low': d['difference low'], 'difference high': d['difference high'],
                          'reading': reading, 'small group': bool(p['small group'])})
decision_table = pd.DataFrame(decision_rows)
print(decision_table.round(3).to_string(index=False))

### Step 5: avoiding leakage when you cross-validate

**What this does.** Cross-validation splits the development data into folds, trains on some and scores on the rest. It is only honest if the folds are independent. News has the same story under different headlines and sources, and if copies of one story fall in the training and the scoring folds, the model is scored on text it has already seen, and the score is too high. This step compares, for the smallest group, two ways of making the folds:

- **Stratified folds** (`StratifiedKFold`): each fold keeps the label proportions, but copies of one story can land in different folds.
- **Grouped stratified folds** (`StratifiedGroupKFold`): label proportions are kept and all rows with the same `DUPLICATE_KEY` stay together in one fold.

It also counts how many development rows share their key with another row.

**What it shows.** How much the way the folds are made changes the score, here and now. Read the count first. If rows share a key and grouping lowers the score clearly, the plain score was inflated. If no rows share a key, as in this example, every row is its own group, and the two scores differ only because the rows fell into folds differently: a gap of a point or two is that chance, not leakage.

**What it does not show.** Leakage you did not name a key for: the same author, the same event, a time pattern, or the same article under a different headline. Naming those is a judgment about your data. It also checks only the folds inside the development rows, not whether a story appears in both the development and the test rows.

In [ ]:
small = development[development[GROUP_COLUMN] == smallest_group].reset_index(drop=True)
key = small[DUPLICATE_KEY].fillna('').astype(str).str.lower().str.strip()
shared = int(key.duplicated(keep=False).sum())
plain = cross_val_score(new_model(), small['_text'], small[LABEL], scoring='f1_macro',
                        cv=StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED))
grouped = cross_val_score(new_model(), small['_text'], small[LABEL], scoring='f1_macro', groups=key,
                          cv=StratifiedGroupKFold(N_FOLDS, shuffle=True, random_state=SEED))
leakage = pd.Series({
    f'rows of {smallest_group} that share a key with another row': shared,
    'macro F1, stratified folds': plain.mean(),
    'macro F1, grouped stratified folds': grouped.mean(),
})
print(leakage.round(3).to_string())

### Step 6: the small-group problem

**What this does.** It takes the smallest group and trains on only a fraction of its data (10, 25, 50 and 100 percent, `TRAIN_FRACTIONS`), in two designs: a **separate** model on that fraction alone, and a **pooled** model on that fraction plus all of the other groups' data. Each is scored on the group's held-out test articles. This shows what happens as a group gets smaller, and whether borrowing from other groups helps.

**What it shows.** How fast a group's results fall as its data shrinks, and whether pooling protects it. Often the separate model collapses first, and the pooled model holds up better, as long as the other groups are close enough to help.

**What it does not show.** That pooling is safe. A pooled model can lean on the big groups and miss what is particular to the small one, so the small group's own test results are the thing to look at, never the pooled average.

In [ ]:
others = development[development[GROUP_COLUMN] != smallest_group]
test_small = held_out[held_out[GROUP_COLUMN] == smallest_group]
curve_rows = []
for fraction in TRAIN_FRACTIONS:
    part = small.sample(frac=fraction, random_state=SEED)
    separate_model = new_model().fit(part['_text'], part[LABEL])
    pooled_model = new_model().fit(pd.concat([part['_text'], others['_text']]), pd.concat([part[LABEL], others[LABEL]]))
    curve_rows.append({'share of the small group used': fraction, 'training articles from it': len(part),
                       'separate model macro F1': macro_f1(test_small[LABEL], separate_model.predict(test_small['_text'])),
                       'pooled model macro F1': macro_f1(test_small[LABEL], pooled_model.predict(test_small['_text']))})
curve = pd.DataFrame(curve_rows)
print(f'smallest group: {smallest_group}')
print(curve.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
shown = intervals.pivot(index='group', columns='model', values='macro F1')
shown.plot.bar(ax=axes[0], yerr=None)
axes[0].set_ylabel('macro F1 on the test articles')
axes[0].set_title('Pooled and separate models, by group')
axes[1].plot(curve['training articles from it'], curve['separate model macro F1'], marker='o', label='separate')
axes[1].plot(curve['training articles from it'], curve['pooled model macro F1'], marker='o', label='pooled')
axes[1].set_xlabel(f'training articles from {smallest_group}')
axes[1].set_title('The smallest group as its data shrinks')
axes[1].legend()
fig.tight_layout()

### Stop and decide: pooled, separate, or pooled with more data for the weakest group

The notebook shows how each design does. It cannot decide which to build.

Questions to ask in the room:

1. For each group, which design looks better, and by how much? Is the difference big enough to matter to the people the group represents, given the interval?
2. Is the weakest group weak because it has little data (Step 6)? If so, can you collect more of it, or is it small for good reason?
3. Can you accept one model that is a little better on average and worse for one group? Who would that group be, and who agreed?
4. What does each design cost to run: building, testing, monitoring and explaining one model or several? What happens when a new group arrives?
5. Did the cross-validation in Step 5 change the picture? Which sources of leakage in your own data have not been named?

Write your choices in the next cell. Leave it blank if you have not decided, and the summary will say so.

In [ ]:
# YOUR CHOICES. The notebook does not fill this in. Plain words are fine.
DESIGN_WE_CHOSE = ''              # pooled, separate, or pooled with more data for the weakest group, and why
WHO_THIS_SERVES_WORST = ''        # the group that does worst under that design, and what you will do about it
DATA_WE_WILL_COLLECT = ''         # more data for which group, from where
LEAKAGE_SOURCES_WE_NAMED = ''     # the keys or patterns you grouped on, and any you could not
WHO_DECIDED = ''                  # the role or body that decided, and where it is written

In [ ]:
EVIDENCE_DIR = Path('evidence')
EVIDENCE_DIR.mkdir(exist_ok=True)

# Small tables of scores. They hold no article.
intervals.round(4).to_csv(EVIDENCE_DIR / 'C-16_group_intervals.csv', index=False)
decision_table.round(4).to_csv(EVIDENCE_DIR / 'C-16_decision_table.csv', index=False)
curve.round(4).to_csv(EVIDENCE_DIR / 'C-16_small_group_curve.csv', index=False)
leakage.round(4).to_frame('value').to_csv(EVIDENCE_DIR / 'C-16_leakage_check.csv')
fig.savefig(EVIDENCE_DIR / 'C-16_comparison.png', dpi=100, bbox_inches='tight')

versions = '\n'.join(f'- {package} {importlib.metadata.version(package)}' for package in PACKAGES)
hashes = '\n'.join(f'- {name}: {value}' for name, value in DATA_SHA256_USED.items())
summary = f"""# C-16 Stratified Model Evaluation: evidence summary

Attach this file, with the CSV and PNG files beside it, to the decision it supports (questions Q-IP-02, Q-IP-04, Q-IP-13, Q-IP-14, Q-IP-19; also Q-PO-12, which the framework does not link, if it fits). It holds scores only, never the data.

## What was run
- Groups: {', '.join(GROUPS)}; development rows {len(development):,}; test rows {len(held_out):,}
- Models: one pooled model and one separate model per group (TF-IDF and logistic regression)
- Intervals: {N_BOOTSTRAP} bootstrap resamples of the test rows; groups under {MIN_GROUP_SIZE} test rows are flagged
- Smallest group: {smallest_group}

## Decision table
{decision_table.round(3).to_string(index=False)}

## Leakage check on {smallest_group}
{leakage.round(3).to_string()}

## Choices made in the stop-and-decide cell
- Design chosen: {DESIGN_WE_CHOSE or 'not written yet'}
- Group served worst, and what we do: {WHO_THIS_SERVES_WORST or 'not written yet'}
- Data we will collect: {DATA_WE_WILL_COLLECT or 'not written yet'}
- Leakage sources named: {LEAKAGE_SOURCES_WE_NAMED or 'not written yet'}
- Who decided: {WHO_DECIDED or 'not written yet'}

## Files beside this summary
C-16_group_intervals.csv, C-16_decision_table.csv, C-16_small_group_curve.csv, C-16_leakage_check.csv, C-16_comparison.png

## Package versions
- Python {platform.python_version()}
{versions}

## Data
SHA-256 of each data file used:
{hashes}
"""
(EVIDENCE_DIR / 'C-16_summary.md').write_text(summary, encoding='utf-8')
print(summary)

## What this does not show, and where to go next

- **It does not say a model is good for a language.** A macro F1 on topic labels is one measure, on one kind of text, from one source. A person who reads that language has to judge whether the model is useful.
- **Groups are not independent.** Languages share loanwords, names and topics, and pooling can help one and hurt another. The test was on four languages out of sixteen.
- **Macro F1 is not comparable from one group to another here.** The languages do not share one set of labels (Luganda and Yoruba each have five of the seven) or one mix of them, so one language's lower macro F1 can come from its labels rather than from the model. Compare the two designs within a group, as the decision table does.
- **Test articles are from the same source as the development ones.** Performance on a different site or period can be lower. See C-21 for watching it over time.
- **The licence covers the release, not the articles.** Keep the files on your machine.
- **Stratifying is a design choice, not a fix.** If a group is poorly served because its data is thin or poorly labelled, a separate model will not repair that.

Where to go next: C-14 Disaggregated Performance Analysis (reporting performance by group with intervals), C-13 Subgroup Fairness Metrics (for decisions about people), C-21 Monitoring and Drift Detection (after launch), and the model card template, whose results table takes the numbers per group.